# LeetCode 1541 — Minimum Insertions to Balance a Parentheses String bằng Python

Link đề: https://leetcode.com/problems/minimum-insertions-to-balance-a-parentheses-string/ · Độ khó: **Medium** · Chủ đề: String, Stack, Greedy, Bracket Sequences

Skeleton LeetCode:

```python
class Solution:
    def minInsertions(self, s: str) -> int:
        ...
```

## 1. Mục tiêu học

1. Hiểu định nghĩa cân bằng **khác thường**: mỗi `'('` phải khớp với **hai `')'` liên tiếp** `'))'`.
2. Viết hàm kiểm tra cân bằng và một baseline **vét cạn chuỗi cân bằng** để có đáp án chuẩn với chuỗi ngắn.
3. Giải bằng **stack** theo gợi ý của đề: xử lý `'))'` liền nhau như một khối, `')'` lẻ thì chèn thêm một `')'`.
4. Rút gọn thành **một lượt duyệt với biến đếm** (`need` = số `')'` còn thiếu), xử lý tinh tế trường hợp `need` lẻ khi gặp `'('`.
5. Nhận ra các bẫy: `')'` không liên tiếp, `')'` lẻ khi stack rỗng phải chèn **2** ký tự, phần `'('` dư ở cuối tốn **2** mỗi cái.

## 2. Đề bài & ví dụ

Cho chuỗi `s` chỉ gồm `'('` và `')'`. Chuỗi **cân bằng** nếu tạo được bằng các luật:

- Chuỗi rỗng cân bằng.
- Nếu `A` cân bằng thì `'(' + A + '))'` cân bằng.
- Nếu `A`, `B` cân bằng thì `A + B` cân bằng.

Nói cách khác, mỗi `'('` phải có một cặp **hai `')'` liên tiếp** đứng sau nó, và mỗi `')'` thuộc đúng một cặp như vậy. Ví dụ `"())"`, `"())(())))"`, `"(())())))"` cân bằng; `")()"`, `"()))"`, `"(()))"` không cân bằng.

Được **chèn** `'('` hoặc `')'` vào bất kỳ đâu. Trả về số lần chèn **ít nhất** để `s` cân bằng.

**Ví dụ 1:** `s = "(()))"` → `1` (chèn thêm `')'` ở cuối → `"(())))"`).

**Ví dụ 2:** `s = "())"` → `0`.

**Ví dụ 3:** `s = "))())("` → `3` (chèn `'('` ở đầu cho `'))'` đầu tiên, chèn `'))'` ở cuối cho `'('` cuối).

**Ràng buộc:** `1 ≤ s.length ≤ 10⁵`, `s` chỉ gồm `'('` và `')'`.

In [1]:
import random
import time
from functools import lru_cache
from itertools import product
from typing import List

examples = [("(()))", 1), ("())", 0), ("))())(", 3)]
for s, want in examples:
    print(f"s = {s!r:10} -> mong đợi {want}")

s = '(()))'    -> mong đợi 1
s = '())'      -> mong đợi 0
s = '))())('   -> mong đợi 3


Hai ví dụ thường gặp khác của bài (kiểm chứng bằng vét cạn/đối chiếu ở mục 12; nên đối chiếu lại đề gốc): `"(((((("` → `12`, `")))))))"` → `5`.

In [2]:
more = [("((((((", 12), (")))))))", 5)]
print(more)

[('((((((', 12), (')))))))', 5)]


## 3. Kiến thức nền

### 3.1 Stack bằng `list`

`append` = đẩy vào đỉnh, `pop` = lấy ra từ đỉnh, `len(st)` = số phần tử; cả hai thao tác `O(1)`. Với bài ngoặc chỉ có một loại `'('`, stack thực chất chỉ cần **một biến đếm**.

In [3]:
st = []
for c in "(()":
    if c == "(":
        st.append(c)
    print(c, st)
st.pop()
print("sau pop:", st, "| còn", len(st), "dấu ( mở")

( ['(']
( ['(', '(']
) ['(', '(']
sau pop: ['('] | còn 1 dấu ( mở


### 3.2 Duyệt bằng chỉ số `while i < n` để "nhìn trước" ký tự kế

Vòng `for c in s` không nhảy được 2 bước. Dùng `while` với `i` để xét `s[i]` và `s[i+1]` (nhớ kiểm tra `i + 1 < n`).

In [4]:
s = "())())"
i, n = 0, len(s)
while i < n:
    if s[i] == ")" and i + 1 < n and s[i + 1] == ")":
        print(f"i={i}: gặp '))' -> nhảy 2")
        i += 2
    else:
        print(f"i={i}: gặp {s[i]!r} -> nhảy 1")
        i += 1

i=0: gặp '(' -> nhảy 1
i=1: gặp '))' -> nhảy 2
i=3: gặp '(' -> nhảy 1
i=4: gặp '))' -> nhảy 2


### 3.3 Kiểm tra cân bằng (theo định nghĩa của bài)

Duyệt và duy trì số `'('` đang mở; gặp `')'` thì **bắt buộc** ký tự kế cũng là `')'` và phải có `'('` đang mở. Cuối cùng không còn `'('` mở.

In [5]:
def is_balanced(s: str) -> bool:
    open_cnt, i, n = 0, 0, len(s)
    while i < n:
        if s[i] == "(":
            open_cnt += 1
            i += 1
        else:
            if i + 1 >= n or s[i + 1] != ")" or open_cnt == 0:
                return False
            open_cnt -= 1
            i += 2
    return open_cnt == 0

for s in ["())", "())(())))", "(())())))", ")()", "()))", "(()))", ""]:
    print(f"{s!r:12} -> {is_balanced(s)}")

'())'        -> True
'())(())))'  -> True
'(())())))'  -> True
')()'        -> False
'()))'       -> False
'(()))'      -> False
''           -> True


### 3.4 `str.replace` để gộp khối `'))'`

`s.replace("))", "}")` thay mọi cặp `'))'` **không chồng lấn từ trái sang phải** bằng một ký tự — sau đó mỗi `')'` còn lại là một `')'` **lẻ**. Chú ý: quét trái → phải là đúng ý ta vì định nghĩa cũng ghép cặp `')'` theo thứ tự xuất hiện.

In [6]:
for s in ["))())(", "()))", ")))"]:
    print(f"{s!r:10} -> {s.replace('))', '}')!r}")

'))())('   -> '}(}('
'()))'     -> '(})'
')))'      -> '})'


### 3.5 Kiểm tra chuỗi con (subsequence)

Để vét cạn, ta cần kiểm tra `s` có phải **chuỗi con** (giữ thứ tự, có thể bỏ ký tự) của chuỗi cân bằng `u` không — vì "chèn ký tự vào `s` để được `u`" ⇔ "`s` là chuỗi con của `u`". Mẹo Python: `it = iter(u); all(c in it for c in s)`.

In [7]:
def is_subseq(s, u):
    it = iter(u)
    return all(c in it for c in s)

print(is_subseq("(()))", "(())))"), is_subseq("))", "()"), is_subseq("", "())"))

True False True


## 4. Ý tưởng nền tảng / trực giác

Mỗi `'('` "đặt hàng" **2** dấu `')'` phải đến **liền nhau**. Tính tay ví dụ 3 `"))())("` (quét trái → phải):

| i | ký tự | đang mở | xử lý | chèn |
|---|-------|---------|-------|------|
| 0–1 | `))` | 0 | cặp `'))'` nhưng không có `'('` mở → chèn `'('` phía trước | 1 |
| 2 | `(` | 1 | mở thêm | 1 |
| 3–4 | `))` | 0 | khớp với `'('` ở i=2 | 1 |
| 5 | `(` | 1 | mở thêm | 1 |
| hết | | 1 | `'('` còn mở cần `'))'` → chèn 2 | 3 |

Còn `')'` lẻ (không có `')'` liền sau)? Ví dụ `"()"`: `')'` ở i=1 lẻ → chèn thêm một `')'` ngay sau nó để thành cặp, rồi khớp với `'('` → tổng 1.

In [8]:
s = "))())("
blocks = s.replace("))", "}")
print("Sau khi gộp '))' thành '}':", blocks, "-> '}' là cặp đủ, ')' là cặp thiếu một")

Sau khi gộp '))' thành '}': }(}( -> '}' là cặp đủ, ')' là cặp thiếu một


## 5. Cách 1 — Brute Force / baseline: vét cạn chuỗi cân bằng

Chuỗi cân bằng gồm `m` dấu `'('` và `2m` dấu `')'` (độ dài `3m`). Sinh **mọi** chuỗi cân bằng theo độ dài tăng dần; chuỗi đầu tiên nhận `s` làm chuỗi con cho đáp án `len(u) − len(s)`. Mỗi ký tự của `s` cần tối đa 2 ký tự chèn, nên chỉ cần xét `3m ≤ 3·len(s)`.

In [9]:
@lru_cache(maxsize=None)
def balanced_of_m(m):
    """Mọi chuỗi cân bằng có đúng m dấu '(' (độ dài 3m)."""
    if m == 0:
        return ("",)
    out = set()
    for a in range(m):                     # u = '(' + A + '))' + B, A có a dấu '(', B có m-1-a
        for A in balanced_of_m(a):
            for B in balanced_of_m(m - 1 - a):
                out.add("(" + A + "))" + B)
    return tuple(sorted(out))

def brute(s):
    m = 0
    while True:
        if 3 * m >= len(s):
            for u in balanced_of_m(m):
                if is_subseq(s, u):
                    return len(u) - len(s)
        m += 1

print("Số chuỗi cân bằng theo m:", [len(balanced_of_m(m)) for m in range(7)])
for s, want in examples:
    print(s, "->", brute(s), "mong đợi", want)

Số chuỗi cân bằng theo m: [1, 1, 2, 5, 14, 42, 132]
(())) -> 1 mong đợi 1
()) -> 0 mong đợi 0
))())( -> 3 mong đợi 3


**Độ phức tạp:** số chuỗi cân bằng tăng theo cấp số mũ (`1, 1, 3, 12, 55, 273, 1428, …`), mỗi chuỗi kiểm tra chuỗi con `O(m)` ⇒ chỉ dùng cho `len(s) ≤ ~7`. Hoàn toàn không dùng được với `10⁵`.

**Nút thắt:** ta thử mọi đáp án thay vì quyết định **cục bộ** ngay khi đọc từng ký tự. Quan sát ở mục 4 cho thấy mỗi tình huống chỉ có một cách xử lý tốt nhất.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** quét trái → phải, đọc theo **khối**: `'('`, `'))'` (cặp đủ), `')'` lẻ (cặp thiếu). Mỗi khối có cách xử lý tham lam duy nhất:

- `'('` ⇒ mở thêm một (`open += 1`).
- Cặp `')'` (đủ hoặc đã được chèn cho đủ):
  - **Nếu** `')'` lẻ **thì** chèn một `')'` (`ans += 1`) cho đủ cặp.
  - **Nếu** có `'('` đang mở **thì** khớp (`open −= 1`); **ngược lại** chèn một `'('` phía trước (`ans += 1`).
- Hết chuỗi ⇒ mỗi `'('` còn mở cần chèn `'))'` (`ans += 2·open`).

**Dạng biến đếm `need`** (không cần nhìn trước): `need` = số `')'` còn thiếu cho các `'('` đang mở.
- Gặp `'('`: `need += 2`; **nếu** `need` lẻ **thì** có một cặp đang dở chỉ có một `')'` — mà `'('` mới chen vào làm hỏng tính liền nhau ⇒ phải chèn `')'` ngay (`ans += 1`, `need −= 1`).
- Gặp `')'`: `need −= 1`; **nếu** `need < 0` **thì** `')'` này không có `'('` ⇒ chèn `'('` (`ans += 1`) và cặp này còn thiếu một `')'` ⇒ `need = 1`.
- Kết thúc: `ans + need`.

**Invariant:** sau mỗi bước, tiền tố đã xét cộng các ký tự đã chèn tạo thành một chuỗi mà **chỉ còn thiếu** đúng `need` dấu `')'` ở cuối (với `need` lẻ nghĩa là đang ở giữa một cặp).

## 7. Các cách tiếp theo

### Cách 2 — Stack + nhìn trước (theo gợi ý của đề)

Từng đoạn:
- `'('` → `push`.
- `')'`: nếu ký tự kế cũng là `')'` thì đây là cặp đủ (nhảy 2); nếu không, chèn một `')'` (`ans += 1`, nhảy 1).
- Có cặp rồi: stack khác rỗng → `pop`; rỗng → chèn `'('` (`ans += 1`).
- Cuối: `ans += 2 * len(stack)`.

In [10]:
def stack_lookahead(s):
    st = []
    ans, i, n = 0, 0, len(s)
    while i < n:
        if s[i] == "(":
            st.append("(")
            i += 1
            continue
        if i + 1 < n and s[i + 1] == ")":    # cặp '))' đủ
            i += 2
        else:                                # ')' lẻ: chèn thêm ')'
            ans += 1
            i += 1
        if st:
            st.pop()
        else:
            ans += 1                         # chèn '(' cho cặp này
    return ans + 2 * len(st)

for s, want in examples + more:
    print(f"{s!r:10} -> {stack_lookahead(s)} (mong đợi {want})")

'(()))'    -> 1 (mong đợi 1)
'())'      -> 0 (mong đợi 0)
'))())('   -> 3 (mong đợi 3)
'(((((('   -> 12 (mong đợi 12)
')))))))'  -> 5 (mong đợi 5)


In [11]:
class SolutionStack:
    def minInsertions(self, s: str) -> int:
        st, ans, i, n = [], 0, 0, len(s)
        while i < n:
            if s[i] == "(":
                st.append(i)
                i += 1
                continue
            if i + 1 < n and s[i + 1] == ")":
                i += 2
            else:
                ans += 1
                i += 1
            if st:
                st.pop()
            else:
                ans += 1
        return ans + 2 * len(st)

print([SolutionStack().minInsertions(s) for s, _ in examples + more])

[1, 0, 3, 12, 5]


**Độ phức tạp:** Time `O(n)`, Space `O(n)` cho stack (worst: toàn `'('`).

### Cách 3 — Thay stack bằng biến đếm `open`

Stack chỉ chứa `'('` giống nhau, nên `len(st)` là đủ — thay bằng một số nguyên. Space còn `O(1)`.

In [12]:
class SolutionCounter:
    def minInsertions(self, s: str) -> int:
        open_cnt, ans, i, n = 0, 0, 0, len(s)
        while i < n:
            if s[i] == "(":
                open_cnt += 1
                i += 1
                continue
            if i + 1 < n and s[i + 1] == ")":
                i += 2
            else:
                ans += 1
                i += 1
            if open_cnt:
                open_cnt -= 1
            else:
                ans += 1
        return ans + 2 * open_cnt

print([SolutionCounter().minInsertions(s) for s, _ in examples + more])

[1, 0, 3, 12, 5]


**Độ phức tạp:** Time `O(n)`, Space `O(1)`.

### Cách 4 — Một lượt, biến `need` (không nhìn trước)

Đây là dạng gọn nhất, đọc từng ký tự một. Phần tinh tế: khi gặp `'('` mà `need` lẻ.

In [13]:
def need_balance(s):
    ans = need = 0
    for c in s:
        if c == "(":
            need += 2
            if need % 2 == 1:        # có một cặp ')' đang dở -> chèn ')' trước '(' mới
                ans += 1
                need -= 1
        else:
            need -= 1
            if need < 0:             # ')' không có '(' -> chèn '('
                ans += 1
                need += 2
    return ans + need

for s, want in examples + more:
    print(f"{s!r:10} -> {need_balance(s)} (mong đợi {want})")

'(()))'    -> 1 (mong đợi 1)
'())'      -> 0 (mong đợi 0)
'))())('   -> 3 (mong đợi 3)
'(((((('   -> 12 (mong đợi 12)
')))))))'  -> 5 (mong đợi 5)


**Độ phức tạp:** Time `O(n)`, Space `O(1)`.

### Cách 5 — Gộp `'))'` bằng `replace` rồi đếm

`t = s.replace("))", "}")`: mỗi `'}'` là cặp đủ, mỗi `')'` còn lại là cặp thiếu một (chèn 1 rồi coi như `'}'`). Sau đó chỉ cần đếm như ngoặc thường với `'('` và `'}'`.

In [14]:
class SolutionReplace:
    def minInsertions(self, s: str) -> int:
        t = s.replace("))", "}")
        ans = t.count(")")                   # mỗi ')' lẻ cần chèn thêm một ')'
        open_cnt = 0
        for c in t:
            if c == "(":
                open_cnt += 1
            elif open_cnt:                   # '}' hoặc ')' (đã được bù) khớp với '('
                open_cnt -= 1
            else:
                ans += 1                     # chèn '('
        return ans + 2 * open_cnt

print([SolutionReplace().minInsertions(s) for s, _ in examples + more])

[1, 0, 3, 12, 5]


**Độ phức tạp:** Time `O(n)`, Space `O(n)` cho chuỗi mới.

## 8. TRACE từng bước

Trace cách 4 trên `s = "()()))"` (đáp án 3):

| i | c | need trước | thao tác | need sau | ans |
|---|---|------------|----------|----------|-----|
| 0 | ( | 0 | +2 | 2 | 0 |
| 1 | ) | 2 | −1 | 1 | 0 |
| 2 | ( | 1 | +2 = 3 lẻ → chèn ')' | 2 | 1 |
| 3 | ) | 2 | −1 | 1 | 1 |
| 4 | ) | 1 | −1 | 0 | 1 |
| 5 | ) | 0 | −1 = −1 < 0 → chèn '(' | 1 | 2 |
| hết | | 1 | ans + need | | **3** |

Hàm `trace_need` in bảng trên kèm **chuỗi kết quả** dựng song song (ký tự chèn viết hoa `[ ]`).

In [15]:
def trace_need(s):
    ans = need = 0
    built = []
    print(f"{'i':>2} | c | need trước | thao tác                     | need sau | ans")
    for i, c in enumerate(s):
        before = need
        note = ""
        if c == "(":
            need += 2
            if need % 2 == 1:
                ans += 1
                need -= 1
                built.append("[)]")
                note = "+2, lẻ -> chèn ')' trước '('"
            else:
                note = "+2"
            built.append("(")
        else:
            need -= 1
            if need < 0:
                ans += 1
                need += 2
                built.append("[(]")
                note = "-1 < 0 -> chèn '(' trước"
            else:
                note = "-1"
            built.append(")")
        print(f"{i:>2} | {c} | {before:>10} | {note:28} | {need:>8} | {ans}")
    built.append("[)]" * need)
    print(f"hết chuỗi: chèn {need} dấu ')' -> tổng {ans + need}")
    print("chuỗi sau khi chèn:", "".join(built))
    return ans + need

trace_need("()()))")

 i | c | need trước | thao tác                     | need sau | ans
 0 | ( |          0 | +2                           |        2 | 0
 1 | ) |          2 | -1                           |        1 | 0
 2 | ( |          1 | +2, lẻ -> chèn ')' trước '(' |        2 | 1
 3 | ) |          2 | -1                           |        1 | 1
 4 | ) |          1 | -1                           |        0 | 1
 5 | ) |          0 | -1 < 0 -> chèn '(' trước     |        1 | 2
hết chuỗi: chèn 1 dấu ')' -> tổng 3
chuỗi sau khi chèn: ()[)]())[(])[)]


3

In [16]:
def trace_stack(s):
    st, ans, i, n = [], 0, 0, len(s)
    while i < n:
        if s[i] == "(":
            st.append(i)
            print(f"i={i}: '(' -> push, stack={st}, ans={ans}")
            i += 1
            continue
        if i + 1 < n and s[i + 1] == ")":
            kind, i = "cặp '))'", i + 2
        else:
            kind, i, ans = "')' lẻ (chèn ')')", i + 1, ans + 1
        if st:
            j = st.pop()
            print(f"{kind}: khớp '(' ở {j}, stack={st}, ans={ans}")
        else:
            ans += 1
            print(f"{kind}: stack rỗng -> chèn '(', ans={ans}")
    print(f"hết chuỗi: còn {len(st)} '(' -> +{2 * len(st)} -> {ans + 2 * len(st)}")
    return ans + 2 * len(st)

trace_stack("))())(")

cặp '))': stack rỗng -> chèn '(', ans=1
i=2: '(' -> push, stack=[2], ans=1
cặp '))': khớp '(' ở 2, stack=[], ans=1
i=5: '(' -> push, stack=[5], ans=1
hết chuỗi: còn 1 '(' -> +2 -> 3


3

## 9. Code nộp LeetCode

Bản nộp chính (cách 4), đúng chữ ký `minInsertions(self, s: str) -> int`:

In [17]:
class Solution:
    def minInsertions(self, s: str) -> int:
        ans = 0        # số ký tự đã chèn
        need = 0       # số ')' còn thiếu cho các '(' đang mở
        for c in s:
            if c == "(":
                need += 2
                if need % 2:           # cặp ')' đang dở bị '(' chen ngang
                    ans += 1           # chèn ')' để hoàn tất cặp đó
                    need -= 1
            else:
                need -= 1
                if need < 0:           # ')' thừa: chèn '(' trước nó
                    ans += 1
                    need = 1           # cặp này còn thiếu một ')'
        return ans + need

for s, want in examples + more:
    got = Solution().minInsertions(s)
    print(f"{s!r:10} -> {got}", "OK" if got == want else "SAI")

'(()))'    -> 1 OK
'())'      -> 0 OK
'))())('   -> 3 OK
'(((((('   -> 12 OK
')))))))'  -> 5 OK


## 10. Vì sao thuật toán đúng

Xét cách 2/3 (đọc theo khối), cách 4 tương đương.

**Phân rã duy nhất.** Trong chuỗi cân bằng, các `')'` được ghép thành cặp **liền nhau** theo thứ tự trái → phải (một `')'` ở đầu cặp luôn đi kèm `')'` ngay sau nó). Vì vậy việc gộp `'))'` từ trái sang phải trong `s` không làm mất phương án tối ưu: một `')'` mà ký tự kế **không** phải `')'` thì trong chuỗi kết quả, giữa nó và ký tự kế **phải** có một `')'` được chèn (hoặc nó là phần sau của cặp với một `')'` chèn trước — cũng tốn 1). ⇒ mỗi `')'` lẻ tốn **ít nhất 1** lần chèn, và chèn đúng 1 là đủ.

**Khớp cặp với `'('`.** Coi mỗi cặp `')'` là một "ngoặc đóng" và `'('` là "ngoặc mở" — bài trở thành ngoặc thường: tham lam khớp với `'('` mở gần nhất. Một cặp đóng khi không còn `'('` mở **bắt buộc** cần một `'('` chèn phía trước (chặn dưới), và chèn 1 là đủ. Cuối chuỗi, mỗi `'('` mở cần đúng 2 dấu `')'` (chặn dưới 2, chèn 2 là đủ).

**Không có tương tác xấu:** các quyết định trên không ảnh hưởng nhau — số `')'` lẻ cố định theo `s`, và với dãy ngoặc thường thì số chèn tối thiểu = (số đóng không khớp) + (số mở dư), đạt bởi tham lam. Tổng hai phần là tối ưu.

**Cách 4 tương đương:** `need` lẻ ⇔ vừa đọc một `')'` mà chưa có `')'` thứ hai; gặp `'('` lúc đó chính là trường hợp `')'` lẻ ⇒ chèn 1. `need < 0` ⇔ cặp đóng không có `'('` ⇒ chèn `'('`.

## 11. Phân tích Time / Space

Gọi `n = len(s)`.

| Cách | Time | Space |
|------|------|-------|
| 1. Vét cạn chuỗi cân bằng | mũ theo `n` | mũ theo `n` |
| 2. Stack + nhìn trước | `O(n)` | `O(n)` |
| 3. Biến đếm + nhìn trước | `O(n)` | `O(1)` |
| 4. Biến `need` một lượt | `O(n)` | `O(1)` |
| 5. `replace` rồi đếm | `O(n)` | `O(n)` |

Mọi cách tuyến tính đều đọc mỗi ký tự đúng một lần (cách 2/3 có thể nhảy 2 ký tự một lúc).

## 12. Bộ test

Ví dụ đề, edge case (một ký tự, toàn `'('`, toàn `')'` chẵn/lẻ, `')('`), đối chiếu với vét cạn trên **mọi** chuỗi độ dài ≤ 7, đối chiếu chéo các cách trên chuỗi ngẫu nhiên (seed cố định), và `n = 10⁵`.

In [18]:
METHODS = {
    "stack_lookahead": stack_lookahead,
    "SolutionStack": lambda s: SolutionStack().minInsertions(s),
    "SolutionCounter": lambda s: SolutionCounter().minInsertions(s),
    "need_balance": need_balance,
    "SolutionReplace": lambda s: SolutionReplace().minInsertions(s),
    "Solution": lambda s: Solution().minInsertions(s),
}
edge = [("(", 2), (")", 2), ("))", 1), (")(", 4), ("()", 1), ("())", 0), ("()))", 2),
        ("())(())))", 0), ("(())())))", 0), (")()", 3), ("()()))", 3)]
for s, want in examples + more + edge:
    if len(s) <= 7:
        assert brute(s) == want, (s, brute(s))
    for name, f in METHODS.items():
        got = f(s)
        assert got == want, (name, s, got, want)
print("ví dụ + edge: OK")

cnt = 0
for n in range(1, 8):
    for t in product("()", repeat=n):
        s = "".join(t)
        ref = brute(s)
        for name, f in METHODS.items():
            assert f(s) == ref, (name, s)
        cnt += 1
print(f"Mọi chuỗi độ dài 1..7 ({cnt} chuỗi): mọi cách khớp vét cạn")

ví dụ + edge: OK
Mọi chuỗi độ dài 1..7 (254 chuỗi): mọi cách khớp vét cạn


In [19]:
rng = random.Random(1541)
for _ in range(1000):
    n = rng.randint(1, 200)
    p = rng.choice([0.33, 0.5, 0.2, 0.7])
    s = "".join("(" if rng.random() < p else ")" for _ in range(n))
    ref = Solution().minInsertions(s)
    for name, f in METHODS.items():
        assert f(s) == ref, (name, s)
print("1000 chuỗi ngẫu nhiên (seed 1541): mọi cách khớp nhau")

1000 chuỗi ngẫu nhiên (seed 1541): mọi cách khớp nhau


In [20]:
rng2 = random.Random(5)
big = "".join(rng2.choice("()") for _ in range(10**5))
big2 = "(" * 50000 + ")" * 50000
big3 = "())" * 33333
for name in ["SolutionStack", "SolutionCounter", "need_balance", "SolutionReplace", "Solution"]:
    t0 = time.perf_counter()
    res = [METHODS[name](x) for x in (big, big2, big3)]
    print(f"{name:15s}: {res} ({time.perf_counter() - t0:.3f}s)")
    assert res[1] == 50000 and res[2] == 0

SolutionStack  : [49505, 50000, 0] (0.020s)


SolutionCounter: [49505, 50000, 0] (0.019s)


need_balance   : [49505, 50000, 0] (0.017s)


SolutionReplace: [49505, 50000, 0] (0.011s)


Solution       : [49505, 50000, 0] (0.015s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Làm như ngoặc thường (một `')'` cho mỗi `'('`)

Quen tay giải bài *Minimum Add to Make Parentheses Valid* (921) ⇒ sai hoàn toàn vì mỗi `'('` cần hai `')'`.

In [21]:
def wrong_normal(s):
    open_cnt = ans = 0
    for c in s:
        if c == "(":
            open_cnt += 1
        elif open_cnt:
            open_cnt -= 1
        else:
            ans += 1
    return ans + open_cnt              # SAI: luật 1-1

for s, want in examples + more:
    print(f"{s!r:10} sai: {wrong_normal(s)} | đúng: {want}")

'(()))'    sai: 1 | đúng: 1
'())'      sai: 1 | đúng: 0
'))())('   sai: 4 | đúng: 3
'(((((('   sai: 6 | đúng: 12
')))))))'  sai: 7 | đúng: 5


Sửa: `need += 2` mỗi `'('`, và xử lý cặp `'))'`.

### Lỗi 2 — Không yêu cầu hai `')'` **liền nhau** (bỏ bước sửa `need` lẻ)

Chỉ đếm `need += 2` / `need −= 1` mà quên trường hợp `'('` chen giữa một cặp `')'` đang dở ⇒ hai `')'` cách nhau vẫn bị coi là một cặp.

In [22]:
def wrong_no_parity(s):
    ans = need = 0
    for c in s:
        if c == "(":
            need += 2                  # SAI: thiếu xử lý need lẻ
        else:
            need -= 1
            if need < 0:
                ans += 1
                need += 2
    return ans + need

for s in ["()()))", ")()))", "()()))("]:
    print(f"{s!r:10} sai: {wrong_no_parity(s)} | đúng: {Solution().minInsertions(s)} | vét cạn: {brute(s)}")

'()()))'   sai: 0 | đúng: 3 | vét cạn: 3
')()))'    sai: 1 | đúng: 4 | vét cạn: 4
'()()))('  sai: 2 | đúng: 5 | vét cạn: 5


Sửa: khi `'('` làm `need` lẻ thì `ans += 1; need −= 1`.

### Lỗi 3 — `')'` lẻ khi không có `'('` chỉ cộng 1

Một `')'` lẻ đứng một mình cần **cả** `'('` phía trước **và** một `')'` nữa ⇒ 2 lần chèn.

In [23]:
def wrong_single_close(s):
    open_cnt, ans, i, n = 0, 0, 0, len(s)
    while i < n:
        if s[i] == "(":
            open_cnt += 1; i += 1; continue
        pair = i + 1 < n and s[i + 1] == ")"
        i += 2 if pair else 1
        if open_cnt:
            open_cnt -= 1
            if not pair:
                ans += 1
        else:
            ans += 1                   # SAI: ')' lẻ + không có '(' phải cộng 2
    return ans + 2 * open_cnt

for s in [")", ")(", ")()"]:
    print(f"{s!r:6} sai: {wrong_single_close(s)} | đúng: {Solution().minInsertions(s)}")

')'    sai: 1 | đúng: 2
')('   sai: 3 | đúng: 4
')()'  sai: 2 | đúng: 3


Sửa: tách bạch hai việc — `')'` lẻ luôn `+1`; không có `'('` luôn `+1`.

### Lỗi 4 — Phần `'('` dư cuối chuỗi chỉ cộng 1 mỗi cái

Mỗi `'('` còn mở cần **hai** `')'`.

In [24]:
def wrong_tail(s):
    open_cnt, ans, i, n = 0, 0, 0, len(s)
    while i < n:
        if s[i] == "(":
            open_cnt += 1; i += 1; continue
        if i + 1 < n and s[i + 1] == ")":
            i += 2
        else:
            ans += 1; i += 1
        if open_cnt:
            open_cnt -= 1
        else:
            ans += 1
    return ans + open_cnt              # SAI: phải là 2 * open_cnt

for s, want in examples + more:
    print(f"{s!r:10} sai: {wrong_tail(s)} | đúng: {want}")

'(()))'    sai: 1 | đúng: 1
'())'      sai: 0 | đúng: 0
'))())('   sai: 2 | đúng: 3
'(((((('   sai: 6 | đúng: 12
')))))))'  sai: 5 | đúng: 5


Sửa: `return ans + 2 * open_cnt`.

### Lỗi 5 — Nhìn trước `s[i + 1]` mà quên kiểm tra biên

Với `')'` ở cuối chuỗi, `s[i + 1]` gây `IndexError`. Ta bắt lỗi để in ra thay vì để chương trình dừng.

In [25]:
def wrong_index(s):
    open_cnt, ans, i = 0, 0, 0
    while i < len(s):
        if s[i] == "(":
            open_cnt += 1; i += 1; continue
        if s[i + 1] == ")":            # SAI: thiếu i + 1 < len(s)
            i += 2
        else:
            ans += 1; i += 1
        if open_cnt:
            open_cnt -= 1
        else:
            ans += 1
    return ans + 2 * open_cnt

for s in ["())", "()", "(()))"]:
    try:
        print(f"{s!r:8} -> {wrong_index(s)}")
    except IndexError as e:
        print(f"{s!r:8} -> IndexError: {e} (đúng phải là {Solution().minInsertions(s)})")

'())'    -> 0
'()'     -> IndexError: string index out of range (đúng phải là 1)
'(()))'  -> IndexError: string index out of range (đúng phải là 1)


Sửa: `if i + 1 < n and s[i + 1] == ")"`, hoặc dùng cách 4 không cần nhìn trước.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Bài ngoặc với "tỷ lệ" khác 1:1** ⇒ dùng biến `need` (số ngoặc đóng còn thiếu) và chú ý **tính liền nhau** (parity).
- Bài ngoặc chỉ một loại ⇒ stack có thể thay bằng **biến đếm**.
- "Số lần chèn tối thiểu" với ngoặc ⇒ tham lam: thiếu mở thì chèn mở ngay, cuối chuỗi bù phần đóng.
- Mẹo `replace("))", "}")` để biến bài về dạng ngoặc thường — chỉ an toàn khi việc ghép cặp trái → phải đúng với định nghĩa.
- Bài cùng họ: *Minimum Add to Make Parentheses Valid* (921), *Valid Parentheses* (20), *Minimum Remove to Make Valid Parentheses* (1249), *Check if a Parentheses String Can Be Valid* (2116).

## 15. Biến thể của bài

1. **Mỗi `'('` cần `k` dấu `')'` liên tiếp** (tổng quát hoá — bài tập 5).
2. **Chỉ được xoá ký tự** thay vì chèn: tìm chuỗi con cân bằng dài nhất — tham lam tương tự nhưng ghép cặp cẩn thận hơn.
3. **Được chèn và xoá, mỗi thao tác chi phí khác nhau**: DP theo (vị trí, số `need`).
4. **Kiểm tra cân bằng** (không chèn): hàm `is_balanced` ở mục 3.3.
5. **Dựng luôn chuỗi kết quả** sau khi chèn tối thiểu (bài tập 4).

In [26]:
def min_insert_dp(s):
    """Biến thể 3 (đơn giản hoá: chỉ chèn, chi phí 1) bằng DP trạng thái need — đối chiếu lời giải tham lam."""
    INF = float("inf")
    dp = {0: 0}                          # need -> chi phí nhỏ nhất
    for c in s:
        nd = {}
        for need, cost in dp.items():
            opts = []
            if c == "(":
                if need % 2:             # cặp dở: buộc chèn ')' trước
                    opts.append((need - 1 + 2, cost + 1))
                else:
                    opts.append((need + 2, cost))
            else:
                if need > 0:
                    opts.append((need - 1, cost))
                else:
                    opts.append((1, cost + 1))
            for k2, v in opts:
                nd[k2] = min(nd.get(k2, INF), v)
        dp = nd
    return min(cost + need for need, cost in dp.items())

print([min_insert_dp(s) for s, _ in examples + more])

[1, 0, 3, 12, 5]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|------|---------|------|-------|--------------|
| 1. Vét cạn | thử mọi chuỗi cân bằng chứa `s` | mũ | mũ | kiểm chứng `n ≤ 7` |
| 2. Stack + nhìn trước | gợi ý đề: cặp `'))'`, `')'` lẻ | `O(n)` | `O(n)` | dễ hiểu nhất |
| 3. Đếm + nhìn trước | stack → số nguyên | `O(n)` | `O(1)` | gọn, an toàn |
| 4. Biến `need` | một lượt, sửa parity | `O(n)` | `O(1)` | **bản nộp**, ngắn nhất |
| 5. `replace` | gộp `'))'` rồi ngoặc thường | `O(n)` | `O(n)` | viết nhanh trong phỏng vấn |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `minInsertions(")))))))")` (7 dấu `')'`) và `minInsertions("(()")` trả về gì?

**Bài 2 (tính tay):** Trace cách 4 trên `s = "(()))(()))()())))"` và cho biết đáp án.

**Bài 3 (điền TODO):** Hoàn thiện `ex3_need(s)` theo cách 4.

**Bài 4 (tự viết lại):** Viết `ex4_build(s)` trả về **một chuỗi cân bằng** thu được bằng cách chèn đúng `minInsertions(s)` ký tự vào `s`.

**Bài 5 (mini challenge):** Tổng quát: mỗi `'('` cần đúng `k` dấu `')'` **liên tiếp**. Viết `ex5_general(s, k)` trả số lần chèn tối thiểu (kiểm tra với `k = 1, 2, 3` bằng vét cạn).

In [27]:
# Bài 3 — điền TODO (chỉ định nghĩa, KHÔNG gọi)
def ex3_need(s: str) -> int:
    ans = need = 0
    for c in s:
        if c == "(":
            # TODO: need += 2; nếu need lẻ thì chèn ')' (ans += 1, need -= 1)
            pass
        else:
            # TODO: need -= 1; nếu need < 0 thì chèn '(' (ans += 1, need = 1)
            pass
    return ans + need

In [28]:
# Bài 4 & 5 — khung tự viết (chỉ định nghĩa, KHÔNG gọi)
def ex4_build(s: str) -> str:
    # TODO
    raise NotImplementedError

def ex5_general(s: str, k: int) -> int:
    # TODO
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Gộp `'))'`: 7 dấu `')'` = 3 cặp + 1 lẻ. (2) Mỗi cặp thiếu `'('` (+1 mỗi cặp), dấu lẻ cần `')'` và `'('` (+2) ⇒ 5. (3) `"(()"`: `'('` thứ hai có một `')'` (thiếu 1), `'('` đầu thiếu 2 ⇒ 3.

**Bài 2.** (1) Dùng `trace_need`. (2) Chú ý `"()"` ở giữa (`')'` lẻ gặp `'('` kế tiếp). (3) Đáp án là 4.

**Bài 3.** (1) Nhánh `'('`: `need += 2`, rồi `if need % 2`. (2) Nhánh `')'`: `need -= 1`, rồi `if need < 0`. (3) Trả `ans + need`.

**Bài 4.** (1) Dựng song song như `trace_need`: chèn `')'` trước `'('` khi `need` lẻ, chèn `'('` trước `')'` khi `need < 0`. (2) Cuối chuỗi thêm `need` dấu `')'`. (3) Kiểm tra bằng `is_balanced`, `is_subseq` và độ dài.

**Bài 5.** (1) Theo dõi `open` (số `'('` chưa đủ, kể cả cái đang dở) và `cur` (số `')'` đã có của nhóm đang dở, `0 ≤ cur < k`). (2) Gặp `'('` khi `cur > 0`: chèn `k − cur` dấu `')'`, đóng nhóm. Gặp `')'` khi `open == 0`: chèn `'('`. (3) Cuối: `ans += open·k − cur`.

## 19. Đáp án tham khảo cho bài tập

In [29]:
# Bài 1
a1 = (Solution().minInsertions(")))))))"), Solution().minInsertions("(()"))
print("Bài 1:", a1)
assert a1 == (5, 3)

# Bài 2
a2 = trace_need("(()))(()))()())))")
assert a2 == 4 == SolutionStack().minInsertions("(()))(()))()())))")

Bài 1: (5, 3)
 i | c | need trước | thao tác                     | need sau | ans
 0 | ( |          0 | +2                           |        2 | 0
 1 | ( |          2 | +2                           |        4 | 0
 2 | ) |          4 | -1                           |        3 | 0
 3 | ) |          3 | -1                           |        2 | 0
 4 | ) |          2 | -1                           |        1 | 0
 5 | ( |          1 | +2, lẻ -> chèn ')' trước '(' |        2 | 1
 6 | ( |          2 | +2                           |        4 | 1
 7 | ) |          4 | -1                           |        3 | 1
 8 | ) |          3 | -1                           |        2 | 1
 9 | ) |          2 | -1                           |        1 | 1
10 | ( |          1 | +2, lẻ -> chèn ')' trước '(' |        2 | 2
11 | ) |          2 | -1                           |        1 | 2
12 | ( |          1 | +2, lẻ -> chèn ')' trước '(' |        2 | 3
13 | ) |          2 | -1                           |        

In [30]:
# Bài 3
def ex3_need_sol(s):
    ans = need = 0
    for c in s:
        if c == "(":
            need += 2
            if need % 2:
                ans += 1
                need -= 1
        else:
            need -= 1
            if need < 0:
                ans += 1
                need = 1
    return ans + need

for n in range(1, 8):
    for t in product("()", repeat=n):
        s = "".join(t)
        assert ex3_need_sol(s) == brute(s)
print("Bài 3: OK")

Bài 3: OK


In [31]:
# Bài 4
def ex4_build_sol(s):
    out, need = [], 0
    for c in s:
        if c == "(":
            if need % 2:
                out.append(")")
                need -= 1
            need += 2
            out.append("(")
        else:
            if need == 0:
                out.append("(")
                need = 2
            need -= 1
            out.append(")")
    out.append(")" * need)
    return "".join(out)

rng4 = random.Random(4)
for _ in range(1000):
    s = "".join(rng4.choice("()") for _ in range(rng4.randint(1, 40)))
    u = ex4_build_sol(s)
    assert is_balanced(u) and is_subseq(s, u) and len(u) - len(s) == Solution().minInsertions(s)
print("Bài 4: OK, ví dụ:", ex4_build_sol("))())("), ex4_build_sol("()()))"))

Bài 4: OK, ví dụ: ())())()) ())())())


In [32]:
# Bài 5
def ex5_general_sol(s, k):
    ans = open_cnt = cur = 0
    for c in s:
        if c == "(":
            if cur:                       # nhóm đang dở bị '(' chen ngang
                ans += k - cur
                cur = 0
                open_cnt -= 1
            open_cnt += 1
        else:
            if open_cnt == 0:             # không có '(' -> chèn '('
                ans += 1
                open_cnt = 1
            cur += 1
            if cur == k:
                cur = 0
                open_cnt -= 1
    return ans + open_cnt * k - cur

@lru_cache(maxsize=None)
def balanced_k(m, k):
    if m == 0:
        return ("",)
    out = set()
    for a in range(m):
        for A in balanced_k(a, k):
            for B in balanced_k(m - 1 - a, k):
                out.add("(" + A + ")" * k + B)
    return tuple(out)

def brute_k(s, k):
    best, m = None, 0
    while best is None:
        for u in balanced_k(m, k):
            if len(u) >= len(s) and is_subseq(s, u):
                best = len(u) - len(s) if best is None else min(best, len(u) - len(s))
        m += 1
    return best

for k in (1, 2, 3):
    for n in range(1, 7):
        for t in product("()", repeat=n):
            s = "".join(t)
            assert ex5_general_sol(s, k) == brute_k(s, k), (s, k)
assert all(ex5_general_sol(s, 2) == Solution().minInsertions(s) for s, _ in examples + more)
print("Bài 5: OK với k = 1, 2, 3 trên mọi chuỗi độ dài ≤ 6")

Bài 5: OK với k = 1, 2, 3 trên mọi chuỗi độ dài ≤ 6


## 20. Template nhớ nhanh + Checklist

```python
class Solution:
    def minInsertions(self, s):
        ans = need = 0                 # need: số ')' còn thiếu
        for c in s:
            if c == "(":
                need += 2
                if need % 2:           # cặp ')' dở bị chen ngang
                    ans += 1; need -= 1
            else:
                need -= 1
                if need < 0:           # ')' không có '('
                    ans += 1; need = 1
        return ans + need
```

**Template "ngoặc tỷ lệ 1 : k":** biến đếm số ngoặc đóng còn thiếu; khi ngoặc mở chen vào giữa nhóm đóng đang dở thì bù ngay phần thiếu; khi đóng thừa thì chèn mở; cuối chuỗi bù phần còn thiếu.

**Checklist trước khi submit:**
- [ ] Mỗi `'('` cần **2** dấu `')'` **liên tiếp**.
- [ ] Xử lý `need` lẻ khi gặp `'('`.
- [ ] `')'` thừa: `ans += 1` và `need = 1` (không phải `need = 0`).
- [ ] Cuối: cộng `need` (hoặc `2·open` ở cách đếm).
- [ ] Nếu nhìn trước `s[i+1]`: kiểm tra `i + 1 < n`.